# 02. Extract Source Text

Extract text from HTML, PDF, DOCX, and HWPX documents and show only short previews. The legacy binary HWP file is signature-validated in Notebook 01 and is clearly marked as requiring a separate converter.

No LLM, API key, or paid token is required.

In [1]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'data' / 'source_manifest.csv').exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not find data/source_manifest.csv')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
from src.pilot_corpus import load_manifest

manifest = load_manifest(REPO_ROOT)
print(f'Manifest entries: {len(manifest)}')

from src.pilot_corpus import extract_sources

Manifest entries: 16


In [2]:
extraction = extract_sources(manifest, REPO_ROOT)
display(extraction[['file', 'extraction', 'pages', 'text_chars']])

summary = {
    'text extraction succeeded': int((extraction['extraction'] == 'extracted').sum()),
    'legacy HWP conversion required': int(extraction['extraction'].str.contains('conversion required').sum()),
    'extraction failures': int((extraction['extraction'] == 'failed').sum()),
}
display(summary)

assert not (extraction['extraction'] == 'failed').any(), 'One or more extractors failed.'
print('Supported source formats were extracted successfully.')

,file,extraction,pages,text_chars
0,raw/law_go_kr/01_university_rules_current_2026...,extracted,NaN,45191
1,raw/law_go_kr/02_graduate_academic_operation_2...,extracted,NaN,11554
2,raw/law_go_kr/03_graduate_academic_operation_c...,extracted,NaN,11936
3,raw/law_go_kr/04_graduate_degree_conferral_cur...,extracted,NaN,4592
4,raw/law_go_kr/05_graduate_curriculum_guideline...,extracted,NaN,991
5,raw/law_go_kr/06_thesis_qualification_exam_gui...,extracted,NaN,1603
6,raw/law_go_kr/07_research_ethics_regulations_2...,extracted,NaN,10296
7,raw/gnu/08_thesis_writing_guidelines_page_2025...,extracted,NaN,2185
8,raw/gnu/08a_thesis_writing_guidelines_ko_en.hwp,binary HWP validated; conversion required,NaN,0
9,raw/gnu/08b_thesis_template_korean.docx,extracted,NaN,1040


{'text extraction succeeded': 15,
 'legacy HWP conversion required': 1,
 'extraction failures': 0}

Supported source formats were extracted successfully.


In [3]:
display(extraction.loc[extraction['text_chars'] > 0, ['file', 'text_chars', 'preview']])

,file,text_chars,preview
0,raw/law_go_kr/01_university_rules_current_2026...,45191,"경상국립대학교 학칙 [시행 2026.2.27.] [경상국립대학교학칙 제509호, 2..."
1,raw/law_go_kr/02_graduate_academic_operation_2...,11554,경상국립대학교 대학원 학사운영규정 [시행 2024.1.16.] [경상국립대학교학교규...
2,raw/law_go_kr/03_graduate_academic_operation_c...,11936,경상국립대학교 대학원 학사운영규정 [시행 2026.2.27.] [경상국립대학교학교규...
3,raw/law_go_kr/04_graduate_degree_conferral_cur...,4592,경상국립대학교 대학원 학위수여규정 [시행 2026.2.27.] [경상국립대학교학교규...
4,raw/law_go_kr/05_graduate_curriculum_guideline...,991,경상국립대학교 대학원 교육과정 운영지침 [시행 2024.1.16.] [경상국립대학교...
5,raw/law_go_kr/06_thesis_qualification_exam_gui...,1603,경상국립대학교 일반대학원 학위논문 제출자격시험 시행지침 [시행 2021.11.23....
6,raw/law_go_kr/07_research_ethics_regulations_2...,10296,경상국립대학교 연구윤리 규정 [시행 2025.2.28.] [경상국립대학교학교규정 제...
7,raw/gnu/08_thesis_writing_guidelines_page_2025...,2185,자료실<커뮤니티 | 대학원 메인메뉴 바로가기 본문으로 바로가기 경상국립대학교 바로가...
9,raw/gnu/08b_thesis_template_korean.docx,1040,국문 예시 1) 표지의 양식 석(박)사 학위논문 (16pt ) 지도교수 ○ ○ ○ ...
10,raw/gnu/08c_thesis_template_english.docx,1334,영 문 예시 1) 표지의 양식 A Thesis for the Degree of Ma...
